In [ ]:
# ============================================
# CELL 1: SETUP KAGGLE ENVIRONMENT
# ============================================

import os
# Ép Kaggle chỉ nhìn thấy và dùng đúng 1 card T4 để tránh lỗi giao tiếp
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 

!pip install -q transformers==4.46.3 datasets sentencepiece accelerate peft==0.10.0

# THƯ MỤC ĐẦU RA (Lưu model sau khi train)
WORKING_DIR = "/kaggle/working/Summarization"
os.makedirs(WORKING_DIR, exist_ok=True)

print(f"💾 Thư mục Output (để save model): {WORKING_DIR}")

In [ ]:
# ============================================
# CELL 2: SET TRAIN SLICE (30K SAMPLES)
# ============================================

# Thiết lập cứng để vã một lèo 30.000 mẫu tập Train từ đầu đến cuối
current_index = 0
next_index = 30000

print("="*50)
print(f"📊 CẤU HÌNH PHÂN TÁCH DỮ LIỆU CHUẨN (60% - 20% - 20%):")
print(f"Lô dữ liệu tập Train sẽ huấn luyện: train[{current_index}:{next_index}]")
print(f"Tổng số mẫu sẽ chạy: {next_index - current_index} mẫu.")
print("="*50)

In [ ]:
# ============================================
# CELL 3: LOAD BASE MODEL & INIT LORA
# ============================================

import torch
import os
from transformers import T5Tokenizer, AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "VietAI/vit5-large"

# Save Adapter mới ra working
FINAL_MODEL_OUTPUT = os.path.join(WORKING_DIR, "vit5-summarization-adapter")

print("Đang tải Tokenizer và Base Model (Float16)...")
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, legacy=False)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16
)
model.enable_input_require_grads()

# Bỏ phần tìm Adapter cũ, auto khởi tạo cấu trúc LoRA mới tinh
print("🌱 Khởi tạo cấu trúc LoRA mới tinh...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)
model = get_peft_model(model, lora_config)

# Chuyển các tham số train được về float32 để ổn định
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

model.print_trainable_parameters()

In [ ]:
# ============================================
# CELL 4: LOAD DATASET
# ============================================

from datasets import load_dataset

# 1. Tải CUỐN CHIẾU lô dữ liệu Train trực tiếp từ nhánh 'train' gốc (Tổng chốt chặn ở Cell 2 là 30k)
data_slice = f"train[{current_index}:{next_index}]"
print(f"⏳ Đang tải tập Train của VietNews phần: {data_slice} ...")
train_chunk = load_dataset("nam194/vietnews", split=data_slice)

# 2. Tải CỐ ĐỊNH đúng 10.000 mẫu Validation từ nhánh 'validation' gốc theo đúng cơ cấu 60-20-20
print("⏳ Đang tải đúng 10.000 mẫu Validation cố định làm Benchmark...")
val_chunk = load_dataset("nam194/vietnews", split="validation[:10000]")

def preprocess_summarization(examples):
    # Làm sạch dấu gạch dưới
    clean_articles = [doc.replace('_', ' ') for doc in examples["article"]]
    clean_abstracts = [abs.replace('_', ' ') for abs in examples["abstract"]]

    inputs = [f"tóm tắt: {doc}" for doc in clean_articles]
    targets = clean_abstracts

    model_inputs = tokenizer(inputs, max_length=1024, padding="max_length", truncation=True)
    labels = tokenizer(targets, max_length=200, padding="max_length", truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("⏳ Đang Tokenize dữ liệu Train...")
tokenized_train = train_chunk.map(preprocess_summarization, batched=True, remove_columns=train_chunk.column_names)

print("⏳ Đang Tokenize dữ liệu Validation...")
tokenized_val = val_chunk.map(preprocess_summarization, batched=True, remove_columns=val_chunk.column_names)

print("✅ Hoàn tất tiền xử lý chuẩn chỉnh 10.000 mẫu Validation!")

In [ ]:
# ============================================
# CELL 5: TRAIN & SAVE
# ============================================

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

# Checkpoint tạm lưu vào working
OUTPUT_DIR = os.path.join(WORKING_DIR, "vit5-summarization-lora")

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    eval_strategy="epoch", # Đã update cú pháp mới theo warning của Huggingface
    per_device_eval_batch_size=4,
    optim="adamw_torch",
    fp16=True,
    learning_rate=2e-4,
    num_train_epochs=1,
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=10,
    overwrite_output_dir=True,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

print("🚀 Bắt đầu huấn luyện và đánh giá song song lô dữ liệu mới trên Kaggle...")
trainer.train()

# Lưu Adapter hoàn chỉnh ra thư mục /kaggle/working/
trainer.model.save_pretrained(FINAL_MODEL_OUTPUT)
print(f"🎉 Đã hoàn thành! Nhớ tải thư mục {FINAL_MODEL_OUTPUT} về máy nhé!")